# Notebook 2 — K-Means Clustering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indicium15/ml-workshop/blob/main/workshop-2/notebooks/02_KMeans_Clustering.ipynb)

## What is K-Means?

**K-Means** is a clustering algorithm that partitions data into *k* groups by:

1. Placing *k* **centroid** points randomly (or via k-means++ initialisation).
2. Assigning each data point to its nearest centroid.
3. Moving each centroid to the **mean** of its assigned points.
4. Repeating steps 2–3 until the assignments stop changing.

## Choosing k — AIC/BIC and the Elbow Method

K-Means requires you to specify *k* in advance. This notebook compares candidate values of *k* using **AIC** and **BIC**, two information criteria that balance model fit against model complexity:

- Lower AIC/BIC means a better trade-off between fit and complexity.
- Look for the elbow where the curve stops improving sharply: adding more clusters beyond this point gives diminishing returns.
- BIC penalizes extra clusters more strongly than AIC, so it often prefers a simpler solution.

In Notebook 1, we used HCA to *discover* cluster structure. Now we ask: **"Can we formalise those groupings?"** K-Means will produce hard cluster assignments that we can compare with the HCA solution.

---

## Running This Notebook

You can use this notebook in either:

- **Google Colab**: best if you do not have Python installed.
- **Local Jupyter**: best if you already have the workshop folder on your computer.

### In Google Colab

1. Open this notebook from GitHub using the **Open in Colab** button.
2. Run the first setup cell.
3. Wait until it says the libraries and workshop files are loaded.
4. In **Step 1**, either use the pre-loaded sample dataset or upload your own CSV file.
5. Continue running each cell from top to bottom.

If a widget does not appear immediately in Colab, wait for the setup cell to finish, then rerun the current cell.


In [1]:
# SETUP
import sys, os, subprocess, importlib.util
from pathlib import Path

# Works locally from workshop-2/notebooks, from workshop-2, or in Google Colab
# after opening the notebook from GitHub.
REPO_URL = 'https://github.com/indicium15/ml-workshop.git'

def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / 'workshop-2',
        Path.cwd().parent / 'workshop-2',
        Path('/content/ml-workshop/workshop-2'),
        Path('/content/workshop-2'),
    ]
    for candidate in candidates:
        if (candidate / 'utils' / 'data_loader.py').exists():
            return candidate.resolve()
    return None

IN_COLAB = 'google.colab' in sys.modules
WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None and IN_COLAB:
    print('Workshop files not found in Colab. Cloning the workshop repository...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/ml-workshop'], check=True)
    WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None:
    raise FileNotFoundError(
        'Could not find workshop-2/utils. Locally, start Jupyter from the workshop-2 folder. '
        'In Colab, upload the workshop-2 folder or open the notebook from the GitHub repository.'
    )

if str(WORKSHOP_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSHOP_ROOT))

required_modules = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'scipy': 'scipy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'ipywidgets': 'ipywidgets',
}
missing = [pip_name for module_name, pip_name in required_modules.items()
           if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORKSHOP_ROOT / 'requirements.txt')], check=True)

import ipywidgets as widgets

print(f'Using workshop files from: {WORKSHOP_ROOT}')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.cluster import KMeans
from utils.data_loader import DataLoaderWidget
from utils.preprocessor import PreprocessorWidget
from utils.plotting import (
    plot_cluster_scatter,
    plot_cluster_profile_heatmap,
)


from IPython.display import display, FileLink


def _export_dataframe(df, filename):
    if df is None or len(df) == 0:
        display(widgets.HTML('<span style="color:red">No results are ready to export yet.</span>'))
        return None

    export_dir = WORKSHOP_ROOT / 'exports'
    export_dir.mkdir(exist_ok=True)
    path = export_dir / filename
    df.to_csv(path, index=False)

    display(widgets.HTML(
        f'<span style="color:green">Saved {len(df):,} rows to <code>{path}</code></span>'
    ))
    display(FileLink(str(path)))

    if IN_COLAB:
        try:
            from google.colab import files
            files.download(str(path))
        except Exception as exc:
            display(widgets.HTML(
                f'<span style="color:orange">Saved file, but automatic Colab download did not start: {exc}</span>'
            ))
    return path


def _result_base_frame(index):
    result = pd.DataFrame({'row_index': list(index)})
    if loader.df is None:
        return result

    source = loader.df.loc[index]
    id_cols = [c for c in source.columns if 'id' in c.lower()]
    for col in id_cols:
        result[col] = source[col].to_numpy()
    return result

print('Libraries loaded ✓')


Using workshop files from: /Users/chaitanya/Documents/Coding/ml-workshop/workshop-2
Libraries loaded ✓


### Colab Notes

- The first setup cell may take a minute because it checks for required packages.
- Uploaded files in Colab are temporary. If the runtime disconnects, upload the file again.
- To keep your own completed version, use **File → Save a copy in Drive**.
- If widgets do not display, run the setup cell again, then rerun the current cell.

## Step 1 — Load Data

The sample dataset is already available when you run the setup cell.

To use your own data in Colab:

1. Click the upload control.
2. Choose a `.csv` file from your computer.
3. Wait for the preview table to appear.
4. Select the columns you want to use.
5. Click **Confirm Selection**.

For local Jupyter, you can either upload a CSV or type a path to a CSV file on your machine.

### What the Load Data controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **CSV path** | Points the notebook to a CSV file on disk. The default path loads the workshop sample dataset. | Change this when running locally with your own CSV file. |
| **Load CSV** | Reads the file from **CSV path** and displays a short preview plus column counts. | Click this after editing the path. |
| **Or upload** | Lets you upload a CSV directly in Colab/Jupyter instead of typing a path. | Use this when your CSV is on your computer rather than already in the notebook folder. |
| **Features** | Selects the input columns the later analysis/model will use. Hold Ctrl/Cmd to select multiple columns. | Exclude IDs, labels/targets, free-text notes, or columns you do not want the method to learn from. |
| **Select all** | Selects every column except common ID/target columns. | Useful as a starting point, then remove anything inappropriate. |
| **Numeric only** | Selects only integer/float columns. | Useful for a simpler first run or for methods where you want to avoid categorical encoding. |
| **Confirm Selection** | Saves the selected features for later cells. | Click this before preprocessing or running any analysis. |

In [ ]:
# LOAD DATA
recommended_features = [
    'lecture_attendance_rate',
    'tutorial_attendance_rate',
    'tutorial_participation_score',
    'office_hours_visits',
    'lms_logins_per_week',
    'assignment_completion_rate',
    'avg_weekly_study_hours',
    'self_reported_stress_level',
    'extracurricular_hours_per_week',
    'part_time_work_hours',
]
loader = DataLoaderWidget(
    show_label_selector=False,
    default_feature_columns=recommended_features,
)
loader.display()

## Step 2 — Preprocessing

> **Scaling is required for K-Means.** K-Means uses Euclidean distance, so features on larger scales will dominate. Always scale your data before clustering.

StandardScaler is selected and cannot be disabled.

### What the Preprocessing controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **Scaling** | Rescales numeric features before K-Means. This notebook keeps scaling enabled because K-Means is distance-based. | Use `standard` for most runs; try `minmax` when all features have meaningful lower/upper bounds. |
| **Categorical enc** | Converts text/category columns into numbers. `label` assigns each category an integer; `onehot` creates one column per category; `drop` removes categorical columns. | Use `onehot` when categories have no natural order. Use `drop` if a categorical column is not useful for clustering. |
| **Random state** | Sets the seed used by reproducible preprocessing steps. | Keep the default when comparing cluster settings; change it only when checking robustness. |
| **Apply Preprocessing** | Builds the scaled feature matrix used by the K-Means cells. | Click after changing selected columns or preprocessing settings. |



In [3]:
# HYPERPARAMETERS
preprocessor = PreprocessorWidget(
    source_loader=loader,
    y=None,
    force_scale=True,
    clustering_mode=True,
)
preprocessor.display()


## Step 2b — Why Normalisation Matters for K-Means

K-Means assigns every point to its **nearest centroid** using Euclidean distance.
If features are on different scales, those with the largest numerical range dominate
the distance calculation - effectively making all other features invisible.

The sample dataset contains two deliberately paired column groups:

| Without scaling — dominates | With scaling — treated equally |
|---|---|
| `total_study_minutes_per_week` (0-2400) | `avg_weekly_study_hours` (0-40) |
| `cumulative_lms_sessions_per_semester` (0-300) | `lms_logins_per_week` (0-20) |

Run the cell below to see **both** the scale chart **and** a side-by-side cluster comparison
(same *k*, same data - only scaling differs). The unscaled version will produce clusters
that are essentially just bands along the `total_study_minutes_per_week` axis.

### What the Normalisation controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **k for demo** | Sets the number of clusters used only in the normalisation demonstration. | Try a few values to see how raw vs scaled features change the clustering behavior. |
| **Show Normalisation Effect** | Shows feature ranges and compares K-Means on raw data versus scaled data. | Run this before the main K-Means model to build intuition for why scaling matters. |



In [4]:
# OUTPUT
from sklearn.preprocessing import StandardScaler
from utils.plotting import plot_feature_scales, plot_normalisation_comparison

_norm_out = widgets.Output()
_norm_btn = widgets.Button(
    description='Show Normalisation Effect',
    button_style='warning',
    icon='bar-chart',
)
_norm_k = widgets.IntSlider(
    value=4, min=2, max=8, step=1,
    description='k for demo:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='320px'),
)

def _show_norm_effect(_btn):
    _norm_out.clear_output()
    with _norm_out:
        try:
            from sklearn.cluster import KMeans as _KM
            df_raw = loader.df
            if df_raw is None:
                display(widgets.HTML('<span style="color:red">Load data first.</span>'))
                return

            # ── Feature scale chart ──────────────────────────────────────────
            display(widgets.HTML('<b>Feature value ranges (raw, unscaled):</b>'))
            fig0 = plot_feature_scales(df_raw)
            plt.show()

            # ── Numeric-only subsets for the direct comparison ───────────────
            exclude = {'student_id', 'performance_band'}
            num_cols = [
                c for c in df_raw.select_dtypes(include='number').columns
                if c not in exclude
            ]
            X_raw = df_raw[num_cols].fillna(0)

            scaler = StandardScaler()
            X_scaled_arr = scaler.fit_transform(X_raw)
            import pandas as _pd
            X_scaled = _pd.DataFrame(X_scaled_arr, columns=num_cols)

            k = _norm_k.value
            km_raw = _KM(n_clusters=k, n_init=10, random_state=42).fit(X_raw)
            km_scaled = _KM(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)

            display(widgets.HTML(
                f'<b>K-Means (k={k}): cluster scatter — raw vs normalised</b><br>'
                '<span style="color:#555;font-size:0.85em">Both plots show the same students '
                'projected via PCA. Only the scaling differs.</span>'
            ))
            fig1 = plot_normalisation_comparison(
                X_raw, X_scaled,
                km_raw.labels_, km_scaled.labels_,
            )
            plt.show()

            # ── Dominant feature check ───────────────────────────────────────
            import numpy as _np
            ranges = X_raw.max() - X_raw.min()
            top_feat = ranges.idxmax()
            display(widgets.HTML(
                f'<div style="background:#fff3cd;border:1px solid #ffc107;'
                f'padding:8px;border-radius:4px;margin-top:6px">'
                f'⚠ <b>Without scaling</b>, the largest-range feature '
                f'(<code>{top_feat}</code>, range = {ranges[top_feat]:.0f}) '
                f'contributes <b>{ranges[top_feat]/ranges.median():.0f}× more</b> '
                f'to Euclidean distance than the median feature (range = {ranges.median():.1f}).'
                f'</div>'
            ))

        except Exception as exc:
            import traceback
            display(widgets.HTML(f'<span style="color:red">Error: {exc}<br>'
                                 f'<pre>{traceback.format_exc()}</pre></span>'))

_norm_btn.on_click(_show_norm_effect)
display(widgets.VBox([
    _norm_k,
    _norm_btn,
    _norm_out,
]))

## Step 3 — AIC/BIC Elbow Method

Run K-Means for a range of *k* values and compare **AIC** and **BIC**.

Look for the elbow point where AIC/BIC stop improving sharply. Lower values are better, but a good workshop choice should also remain interpretable.

### What the AIC/BIC controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **K min** | Sets the smallest number of clusters to test. | Usually start at 1 or 2. Starting at 1 helps show the first big improvement clearly. |
| **K max** | Sets the largest number of clusters to test. | Increase it if the elbow has not appeared yet; keep it modest to avoid slow or noisy comparisons. |
| **Run AIC/BIC** | Trains K-Means once for each k in the range and plots AIC/BIC. | Click after changing the k range or preprocessing settings. |
| **Export AIC/BIC** | Saves the table of k, inertia, AIC, and BIC as CSV. | Use it when you want to include the model-selection results in notes or reports. |


In [5]:
# HYPERPARAMETERS
_criteria_out = widgets.Output()

_k_min = widgets.IntText(value=1, description='K min:', layout=widgets.Layout(width='120px'),
                         style={'description_width': '50px'})
_k_max = widgets.IntText(value=10, description='K max:', layout=widgets.Layout(width='120px'),
                         style={'description_width': '50px'})
_criteria_btn = widgets.Button(description='Run AIC/BIC', button_style='primary', icon='play')
_export_criteria_btn = widgets.Button(description='Export AIC/BIC', button_style='', icon='download')

_aic_values = []
_bic_values = []
_k_range_used = []
_k_criteria_df = None

def _kmeans_information_criteria(model, X):
    X_arr = X.to_numpy() if hasattr(X, 'to_numpy') else np.asarray(X)
    n_samples, n_features = X_arr.shape
    sse = max(float(model.inertia_), np.finfo(float).eps)
    variance = max(sse / (n_samples * n_features), np.finfo(float).eps)
    log_likelihood = -0.5 * n_samples * n_features * (np.log(2 * np.pi * variance) + 1)
    n_parameters = model.n_clusters * n_features + 1
    aic = 2 * n_parameters - 2 * log_likelihood
    bic = np.log(n_samples) * n_parameters - 2 * log_likelihood
    return aic, bic

def _run_criteria(_btn):
    global _aic_values, _bic_values, _k_range_used, _k_criteria_df
    _criteria_out.clear_output()
    with _criteria_out:
        try:
            X_scaled = preprocessor.X_scaled
            if X_scaled is None:
                display(widgets.HTML('<span style="color:red">Run preprocessing first.</span>'))
                return

            k_min = max(1, _k_min.value)
            k_max = min(20, _k_max.value)
            if k_max < k_min:
                display(widgets.HTML('<span style="color:red">K max must be greater than or equal to K min.</span>'))
                return

            k_range = range(k_min, k_max + 1)
            _k_range_used = list(k_range)

            display(widgets.HTML('Computing K-Means AIC/BIC for each k...'))

            rows = []
            for k in k_range:
                km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
                km.fit(X_scaled)
                aic, bic = _kmeans_information_criteria(km, X_scaled)
                rows.append({
                    'k': k,
                    'inertia': km.inertia_,
                    'aic': aic,
                    'bic': bic,
                })

            _k_criteria_df = pd.DataFrame(rows)
            _aic_values = _k_criteria_df['aic'].tolist()
            _bic_values = _k_criteria_df['bic'].tolist()

            fig, ax = plt.subplots(figsize=(8, 4.5))
            ax.plot(_k_criteria_df['k'], _k_criteria_df['aic'], 'o-', color='steelblue', linewidth=2, label='AIC')
            ax.plot(_k_criteria_df['k'], _k_criteria_df['bic'], 's-', color='darkorange', linewidth=2, label='BIC')
            ax.set_xlabel('Number of clusters (k)')
            ax.set_ylabel('Information criterion (lower is better)')
            ax.set_title('AIC/BIC Elbow Method')
            ax.set_xticks(_k_range_used)
            ax.grid(alpha=0.2)
            ax.legend()
            plt.tight_layout()
            plt.show()

            display(widgets.HTML('<b>AIC/BIC by k:</b>'))
            display(_k_criteria_df.style.format({'inertia': '{:.1f}', 'aic': '{:.1f}', 'bic': '{:.1f}'}))
            display(widgets.HTML(
                'Use the bend in the AIC/BIC curves as the elbow. If AIC and BIC disagree, '
                'BIC is the more conservative choice because it penalizes extra clusters more strongly.'
            ))

        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

def _export_k_criteria(_btn):
    with _criteria_out:
        _export_dataframe(_k_criteria_df, 'kmeans_aic_bic_by_k.csv')

_criteria_btn.on_click(_run_criteria)
_export_criteria_btn.on_click(_export_k_criteria)
display(widgets.VBox([
    widgets.HTML('<h3>Step 3 — AIC/BIC Elbow Method</h3>'),
    widgets.HBox([_k_min, _k_max, _criteria_btn, _export_criteria_btn]),
]))
display(_criteria_out)


Output()

## Step 4 — Train K-Means

Choose your hyperparameters based on the AIC/BIC elbow above, then train the model.

### What the K-Means controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **N clusters (k)** | Sets the number of clusters the final model will create. | Choose this from the AIC/BIC elbow results and from what is meaningful for your use case. |
| **Init method** | Chooses how starting centroids are initialized. `k-means++` spreads them out; `random` places them randomly. | Keep `k-means++` for most runs. Use `random` only to demonstrate sensitivity to initialization. |
| **Max iterations** | Maximum centroid update rounds for a single run. | Increase it if the model fails to converge; lower it only for faster demonstrations. |
| **N restarts** | Number of different initializations to try. The best result is kept. | Increase it for more stable clusters, especially with larger k or random initialization. |
| **Random state** | Controls reproducibility for centroid initialization. | Keep fixed when comparing settings; change it to test cluster stability. |
| **Train Model** | Fits K-Means, stores cluster labels, and displays cluster sizes and centroids. Use the export button to save the labels as CSV. | Click after choosing k and hyperparameters. |

### How to read the outputs after training

After you click **Train Model**, the notebook creates several visual checks to help you decide whether the chosen clustering is useful and interpretable.

| Output | What it represents | What to look for |
|---|---|---|
| **Training summary** | Reports the selected *k*, model inertia, AIC, and BIC. | Lower AIC/BIC means a better fit-complexity trade-off. Inertia is still shown as the raw within-cluster SSE. |
| **AIC/BIC elbow plot** | Re-displays the information-criteria plot and highlights the *k* used for the final model. | Check whether the selected *k* is near the bend in the curve, where adding more clusters gives smaller improvements. |
| **PCA cluster scatter plot** | Projects the scaled feature space down to two PCA dimensions so the clusters can be viewed on a 2D chart. | Look for visible separation between colors, overlap between clusters, and outlying points. Because this is a 2D projection, overlapping points do not always mean the clusters overlap in the full feature space. |
| **Cluster sizes table** | Counts how many rows were assigned to each cluster. | Watch for very tiny clusters, which may represent outliers, an overly large *k*, or a meaningful but small subgroup. |
| **Cluster centroid heatmap** | Shows the average scaled feature profile for each cluster. Positive values are above the dataset average; negative values are below it. | Use this to name and explain the clusters. Focus on the strongest positive and negative values in each row rather than small differences close to 0. |

Use the plots together rather than relying on one chart. A good *k* should usually have a reasonable AIC/BIC elbow, clusters that are not extremely imbalanced, and centroid profiles that make sense for the problem you are studying.


In [ ]:
# HYPERPARAMETERS
_km_out = widgets.Output()
_km_model = None
_km_labels = None
_km_results_df = None

_k_slider = widgets.IntSlider(
    value=4, min=2, max=10, step=1,
    description='N clusters (k):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='450px'),
)
_init_dd = widgets.Dropdown(
    options=['k-means++', 'random'],
    value='k-means++',
    description='Init method:',
    style={'description_width': '150px'},
)
_max_iter_slider = widgets.IntSlider(
    value=300, min=100, max=500, step=50,
    description='Max iterations:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='450px'),
)
_n_init_slider = widgets.IntSlider(
    value=10, min=5, max=20, step=1,
    description='N restarts:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='450px'),
)
_seed_input = widgets.IntText(
    value=42,
    description='Random state:',
    layout=widgets.Layout(width='200px'),
    style={'description_width': '110px'},
)
_train_btn = widgets.Button(
    description='Train Model',
    button_style='success',
    icon='play',
)
_export_km_btn = widgets.Button(
    description='Export Cluster Labels',
    button_style='',
    icon='download',
)

def _build_kmeans_results(labels):
    result = _result_base_frame(preprocessor.X_scaled.index)
    result['kmeans_cluster'] = labels
    result['k'] = _k_slider.value
    result['init_method'] = _init_dd.value
    result['random_state'] = _seed_input.value
    return result

def _plot_selected_information_criteria(selected_k):
    if _k_criteria_df is None or _k_criteria_df.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(_k_criteria_df['k'], _k_criteria_df['aic'], 'o-', color='steelblue', linewidth=2, label='AIC')
    ax.plot(_k_criteria_df['k'], _k_criteria_df['bic'], 's-', color='darkorange', linewidth=2, label='BIC')
    if selected_k in set(_k_criteria_df['k']):
        selected = _k_criteria_df.loc[_k_criteria_df['k'] == selected_k].iloc[0]
        ax.axvline(x=selected_k, color='tomato', linestyle='--', label=f'k = {selected_k}')
        ax.scatter([selected_k], [selected['aic']], color='tomato', s=80, zorder=5)
        ax.scatter([selected_k], [selected['bic']], color='tomato', s=80, zorder=5)
    ax.set_xlabel('Number of clusters (k)')
    ax.set_ylabel('Information criterion (lower is better)')
    ax.set_title('AIC/BIC Elbow Method')
    ax.set_xticks(_k_criteria_df['k'].tolist())
    ax.grid(alpha=0.2)
    ax.legend()
    plt.tight_layout()
    plt.show()

def _train_kmeans(_btn):
    global _km_model, _km_labels, _km_results_df
    _km_out.clear_output()
    with _km_out:
        try:
            X_scaled = preprocessor.X_scaled
            if X_scaled is None:
                display(widgets.HTML('<span style="color:red">Run preprocessing first.</span>'))
                return

            k = _k_slider.value
            _km_model = KMeans(
                n_clusters=k,
                init=_init_dd.value,
                max_iter=_max_iter_slider.value,
                n_init=_n_init_slider.value,
                random_state=_seed_input.value,
            )
            _km_labels = _km_model.fit_predict(X_scaled)
            aic, bic = _kmeans_information_criteria(_km_model, X_scaled)
            _km_results_df = _build_kmeans_results(_km_labels)

            display(widgets.HTML(
                f'<span style="color:green">K-Means trained — k={k}, '
                f'inertia={_km_model.inertia_:.1f}, AIC={aic:.1f}, BIC={bic:.1f}</span>'
            ))

            # AIC/BIC plot with selected k highlighted
            _plot_selected_information_criteria(k)

            # 2D scatter
            fig3 = plot_cluster_scatter(
                X_scaled, _km_labels,
                method='PCA',
                title=f'K-Means Clusters — PCA projection (k={k})',
            )
            plt.show()

            # Cluster sizes
            counts = pd.Series(_km_labels).value_counts().sort_index().rename('n_students')
            display(widgets.HTML('<b>Cluster sizes:</b>'))
            display(counts.to_frame())

            # Centroid heatmap (back in original feature space)
            feat_names = list(X_scaled.columns)
            centroid_df = pd.DataFrame(
                _km_model.cluster_centers_,
                columns=feat_names,
                index=[f'Cluster {i}' for i in range(k)],
            )
            display(widgets.HTML('<b>Cluster centroids (scaled feature space):</b>'))
            fig4 = plot_cluster_profile_heatmap(centroid_df, title='Cluster Centroids')
            plt.show()

            display(widgets.HTML('Use <b>Export Cluster Labels</b> to save one row per student with the assigned cluster.'))

        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

def _export_kmeans_results(_btn):
    with _km_out:
        _export_dataframe(_km_results_df, 'kmeans_cluster_labels.csv')

_train_btn.on_click(_train_kmeans)
_export_km_btn.on_click(_export_kmeans_results)
display(widgets.VBox([
    widgets.HTML('<h3>Step 4 — K-Means Hyperparameters</h3>'),
    _k_slider,
    _init_dd,
    _max_iter_slider,
    _n_init_slider,
    _seed_input,
    widgets.HBox([_train_btn, _export_km_btn]),
]))
display(_km_out)


Output()

## After the Workshop: Reusing This K-Means Notebook

Use this notebook when you want a practical clustering model that assigns every row to one of *k* groups.

K-Means is useful when:

- you already have a rough idea of how many groups you want
- you want simple cluster assignments
- you want to compare clusters against a known column
- you want a repeatable segmentation method

### What to Save

For your own analysis notes, save:

- the feature columns used
- the scaling method
- the selected value of *k*
- the AIC/BIC elbow plot
- the exported cluster label CSV
- the cluster size table
- the cluster profile heatmap
- any comparison table with a known column
